In [1]:
import pandas as pd
import re
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_distances

In [2]:
data = pd.read_csv("data/additional_features.csv")
data

,id,Rank,Title,Primary Artists,Featuring Artists,Date,Danceability,Energy,Loudness,Speechiness,...,lyrics,detected_languages,num_languages,language_confidences,artist_id,explicit,track_popularity,artist_followers,artist_genres,artist_popularity
0,47EiUVwUp4C9fGccaPuUCS,1,dákiti,"bad bunny, jhayco",NaN,26/11/2020,0.731,0.573,-10059.0,0.054,...,"Baby, ya yo me enteré, se nota cuando me ve'\n...",SPANISH,1,SPANISH: 1.0,4q3ewBCX7sLwd24euuV69X,True,78,102185553.0,"reggaeton, trap latino, urbano latino, latin",95.0
1,7igeByaBM0MgGsgXtNxDJ7,1,positions,ariana grande,NaN,29/10/2020,0.736,0.802,-4759.0,0.086,...,Heaven sent you to me\nI'm just hopin’ I don't...,ENGLISH,1,ENGLISH: 1.0,66CXWjxzNUsdJxJ2JdwvnR,True,0,107261237.0,pop,90.0
2,3qQbCzHBycnDpGskqOWY0E,1,ella baila sola,"eslabon armado, peso pluma",NaN,29/05/2023,0.668,0.758,-5176.0,0.033,...,"Compa, ¿qué le parece esa morra?\nLa que anda ...",SPANISH,1,SPANISH: 1.0,0XeEobZplHxzM9QzFQWLiR,False,73,6434994.0,"corridos tumbados, sad sierreño, corrido, sier...",75.0
3,0Yde1VrWbGF2Tus2AQhMuT,1,circles,post malone,NaN,06/09/2019,0.703,0.756,-3407.0,0.038,...,"Oh, oh, oh\nOh, oh, oh\nOh, oh, oh, oh, oh\n...",ENGLISH,1,ENGLISH: 1.0,246dkjvS1zLTtiykXe5h60,False,0,47686394.0,NaN,85.0
4,0TK2YIli7K1leLovkQiNik,1,señorita,"camila cabello, shawn mendes",NaN,25/07/2019,0.759,0.540,-6039.0,0.029,...,I love it when you call me señorita\nI wish I ...,ENGLISH,1,ENGLISH: 1.0,7n2wHs1TKAczGzO7Dd2rGr,False,79,45665812.0,NaN,81.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9156,5FOCkGt7bp22nBVfQ4WSIT,200,indecente,anitta,NaN,19/04/2018,0.874,0.602,-5118.0,0.127,...,"Una mirada te delata, un perfume que te atrapa...",SPANISH,1,SPANISH: 1.0,7FNnA9vBm6EKceENgCGRMb,False,48,13245754.0,"brazilian pop, funk pop, brazilian funk, funk ...",78.0
9157,7zQirOExB0VR8yWUOqYeio,200,dolla sign slime (feat. megan thee stallion),lil nas x,megan thee stallion,27/09/2021,0.923,0.533,-6016.0,0.262,...,Day-Day-Daytrip took it to ten (\nHey\n)\n\nY'...,ENGLISH,1,ENGLISH: 1.0,7jVv8c5Fj3E9VhNjxT4snq,True,50,12311640.0,NaN,71.0
9158,5EgYDyMDCi8X0zDydTF7Rb,200,heartbeat (bts world origil soundtrack),bts,NaN,03/07/2019,0.437,0.716,-4306.0,0.050,...,NaN,NaN,0,NaN,3Nrfpe0tUJi4K4DXYWgMUX,False,0,81105517.0,k-pop,84.0
9159,01vv2AjxgP4uUyb8waYO5Y,200,morph,twenty one pilots,NaN,16/10/2018,0.734,0.607,-7249.0,0.081,...,Can't stop thinking about if and when I die\nF...,ENGLISH,1,ENGLISH: 1.0,3YQKmKGau1PzlVlkL1iodx,False,65,25743995.0,NaN,83.0


In [3]:
seed = np.random.randint(1000)
sample = data.sample(n=10, random_state=seed)

print(f"Seed = {seed}")
sample[["Title", "Primary Artists", "Featuring Artists"]]

Seed = 678


,Title,Primary Artists,Featuring Artists
6936,confessions of a dangerous mind,logic,NaN
9041,what i like about you (feat. theresa rex),jos blue,theresa rex
5509,papi's home,drake,NaN
3115,kitsch,ive,NaN
1089,look what you made me do,taylor swift,NaN
5176,gangstas,pop smoke,NaN
6377,not ok,"chelsea cutler, kygo",NaN
1269,chained to the rhythm,"katy perry, skip marley",NaN
5009,welcome to the party,pop smoke,NaN
1183,stay gold,bts,NaN


In [4]:
features = ["Danceability", "Energy", "Loudness", "Speechiness", "Acousticness", "Instrumentalness", "Valence"]

def get_recomendations(song_list, n_recs):
    if song_list.empty:
        return pd.DataFrame(columns=['Title', 'Primary Artists', 'Featuring Artists', 'Similarity'])

    # impute & scale using full dataset for consistency
    impute_vals = data[features].median()
    X_full_imputed = data[features].fillna(impute_vals)
    scaler = StandardScaler().fit(X_full_imputed)
    X_scaled = pd.DataFrame(scaler.transform(X_full_imputed), index=X_full_imputed.index, columns=features)

    liked_idx = song_list.index
    liked_vectors = X_scaled.loc[liked_idx].values  # shape (m_liked, n_features)

    # pairwise distances between each liked song and every song in dataset
    dists = cosine_distances(liked_vectors, X_scaled.values)  # shape (m_liked, n_songs)
    sims_matrix = 1 - dists  # convert distances to similarities

    # aggregate similarities across liked songs
    sims = np.median(sims_matrix, axis=0)
    
    scores = pd.DataFrame({'Similarity': sims}, index=X_scaled.index)
    top = scores.drop(index=liked_idx).sort_values('Similarity', ascending=False).head(n_recs)

    recommendations = data.loc[top.index, ['Title', 'Primary Artists', 'Featuring Artists'] + features].copy()
    recommendations['Similarity'] = top['Similarity']
    return recommendations

In [5]:
mask = data['Primary Artists'].fillna('').str.contains(r'\b(?:slipknot)\b', case=False, regex=True)
liked_songs = data[mask].copy()

if liked_songs.shape[0] > 10:
    liked_songs = liked_songs.sample(10)

liked_songs[["Title", "Primary Artists", "Featuring Artists"] + features]

,Title,Primary Artists,Featuring Artists,Danceability,Energy,Loudness,Speechiness,Acousticness,Instrumentalness,Valence
1420,critical darling,slipknot,NaN,0.297,0.952,-3674.0,0.091,0.001,0.006,0.072
1459,unsainted,slipknot,NaN,0.427,0.946,-2188.0,0.047,0.000,0.025,0.237
2070,nero forte,slipknot,NaN,0.336,0.978,-2358.0,0.085,0.000,0.001,0.496
3381,birth of the cruel,slipknot,NaN,0.432,0.953,-4469.0,0.082,0.000,0.138,0.481
4412,unsainted,slipknot,NaN,0.427,0.944,-2885.0,0.057,0.000,0.016,0.248


In [6]:
recomendations = get_recomendations(liked_songs, 10)
recomendations

,Title,Primary Artists,Featuring Artists,Danceability,Energy,Loudness,Speechiness,Acousticness,Instrumentalness,Valence,Similarity
6778,11 minutes (with halsey feat. travis barker),yungblud,"halsey, travis barker",0.464,0.852,-3804.0,0.067,0.012,0.000,0.233,0.976750
1297,layla,"dj robin, schürze",NaN,0.442,0.922,-2918.0,0.070,0.001,0.000,0.388,0.972784
6506,blah blah blah,armin van buuren,NaN,0.378,0.963,-3911.0,0.068,0.009,0.004,0.160,0.971783
8637,hold on - d&b version,nano,NaN,0.433,0.878,-3097.0,0.051,0.033,0.000,0.386,0.969903
5127,layla,"dj robin, schürze",NaN,0.448,0.921,-3468.0,0.069,0.001,0.000,0.375,0.967759
5526,emo girl (feat. willow),machine gun kelly,willow,0.412,0.881,-3502.0,0.087,0.000,0.000,0.359,0.965850
3017,emo girl (feat. willow),machine gun kelly,willow,0.410,0.881,-3502.0,0.087,0.000,0.000,0.357,0.965737
852,protect the land,system of a down,NaN,0.432,0.886,-3991.0,0.034,0.000,0.000,0.248,0.964967
8356,get it right (feat. mø),diplo,mø,0.456,0.833,-4242.0,0.096,0.108,0.000,0.261,0.961608
1070,lux æterna,metallica,NaN,0.418,0.995,-2974.0,0.080,0.000,0.001,0.243,0.959629
